In [1]:
import requests
import pandas as pd
from parsel import Selector


class BayutScraper:

    def __init__(self):
        self.base_url = "https://www.bayut.bh"
        self.current_page = 1
        self.found_count = 0
        self.data = []

    def get_page(self, url):

        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers)

        return Selector(text=response.text)

    def scrape_details(self, url):

        selector = self.get_page(url)

        self.found_count += 1

        item = {}

        item["url"] = url

        ref_text = selector.xpath(
            '//span[@aria-label="Reference"]//text()'
        ).get()

        item["reference_number"] = ref_text

        if ref_text and "ID" in ref_text:
            item["id"] = ref_text.split("ID")[-1].strip()
        else:
            item["id"] = None

        item["purpose"] = selector.xpath(
            '//span[@aria-label="Purpose"]//text()'
        ).get()

        item["title"] = selector.xpath(
            '//h1/text()'
        ).get()

        desc_parts = selector.xpath(
            '//span[@aria-label="Description"]//text() | //div[@aria-label="Property description"]//text()'
        ).getall()

        item["description"] = " ".join(
            [x.strip() for x in desc_parts if x.strip()]
        )

        item["location"] = selector.xpath(
            '//div[@aria-label="Property header"]//text()'
        ).get()

        item["price"] = selector.xpath(
            '//span[@aria-label="Price"]//text()'
        ).get()

        item["currency"] = selector.xpath(
            '//span[@aria-label="Currency"]//text()'
        ).get()

        item["price_per"] = selector.xpath(
            '//span[@aria-label="Frequency"]//text()'
        ).get()

        item["furnished"] = selector.xpath(
            '//span[@aria-label="Furnishing"]//text()'
        ).get()

        amenities = selector.xpath(
            '//span[@aria-label="Amenity"]//text()'
        ).getall()

        item["amenities"] = ", ".join(
            set([a.strip() for a in amenities if a.strip()])
        )

        item["details"] = selector.xpath(
            '//span[@aria-label="Area"]//text()'
        ).get()

        item["agent_name"] = selector.xpath(
            '//span[@aria-label="Agent name"]//text()'
        ).get()

        images = selector.xpath(
            '//picture//img/@src'
        ).getall()

        item["property_image_urls"] = ", ".join(list(set(images)))

        item["number_of_photos"] = len(set(images))

        breadcrumbs = selector.xpath(
            '//div[@aria-label="Breadcrumb"]//span/text()'
        ).getall()

        item["breadcrumb"] = " > ".join(
            [b.strip() for b in breadcrumbs if b.strip()]
        )

        item["property_type"] = selector.xpath(
            '//span[@aria-label="Type"]//text()'
        ).get()

        self.data.append(item)

        print(f"Scraped Property {self.found_count}")

    def parse(self, url):

        selector = self.get_page(url)

        property_urls = selector.xpath(
            '//a[contains(@href, "/en/property/details-")]/@href'
        ).getall()

        for p_url in property_urls:

            if self.found_count >= 1050:
                return

            if not p_url.startswith("http"):
                p_url = self.base_url + p_url

            self.scrape_details(p_url)

    def run(self):

        while self.found_count < 1050:

            url = (
                f"https://www.bayut.bh/en/to-rent/commercial/bahrain/page-{self.current_page}/"
            )

            print(f"\nScraping Page {self.current_page}")

            self.parse(url)

            self.current_page += 1

        df = pd.DataFrame(self.data)

        df.to_csv("bayut_data.csv", index=False)

        print("\nData saved successfully!")
        print(f"Total Records: {len(df)}")


scraper = BayutScraper()
scraper.run()


Scraping Page 1

Scraping Page 2
Scraped Property 1
Scraped Property 2
Scraped Property 3
Scraped Property 4
Scraped Property 5
Scraped Property 6
Scraped Property 7
Scraped Property 8
Scraped Property 9
Scraped Property 10
Scraped Property 11
Scraped Property 12

Scraping Page 3
Scraped Property 13
Scraped Property 14
Scraped Property 15
Scraped Property 16
Scraped Property 17
Scraped Property 18
Scraped Property 19
Scraped Property 20
Scraped Property 21
Scraped Property 22
Scraped Property 23
Scraped Property 24

Scraping Page 4
Scraped Property 25
Scraped Property 26
Scraped Property 27
Scraped Property 28
Scraped Property 29
Scraped Property 30
Scraped Property 31
Scraped Property 32
Scraped Property 33
Scraped Property 34
Scraped Property 35
Scraped Property 36

Scraping Page 5
Scraped Property 37
Scraped Property 38
Scraped Property 39
Scraped Property 40
Scraped Property 41
Scraped Property 42
Scraped Property 43
Scraped Property 44
Scraped Property 45
Scraped Property 46
Scra

In [4]:
pd.read_csv("bayut_data.csv")

,url,reference_number,id,purpose,title,description,location,price,currency,price_per,furnished,amenities,details,agent_name,property_image_urls,number_of_photos,breadcrumb,property_type
0,https://www.bayut.bh/en/property/details-10564...,Bayut - ID105648594,105648594,For Rent,"1 Bedroom Other Commercial For Rent Gufool, Ca...",Discover a unique opportunity with this commer...,"Gufool, Capital Governorate",350,BHD,Yearly,Unfurnished,NaN,110 Sq. M.,Tanmeya Investment,https://images.bayut.bh/thumbnails/213878-800x...,18,Capital Governorate Other Commercial > Gufool ...,Other Commercial
1,https://www.bayut.bh/en/property/details-10564...,Bayut - ID105648594,105648594,For Rent,"1 Bedroom Other Commercial For Rent Gufool, Ca...",Discover a unique opportunity with this commer...,"Gufool, Capital Governorate",350,BHD,Yearly,Unfurnished,NaN,110 Sq. M.,Tanmeya Investment,https://images.bayut.bh/thumbnails/213878-800x...,18,Capital Governorate Other Commercial > Gufool ...,Other Commercial
2,https://www.bayut.bh/en/property/details-10566...,Bayut - ID105667506,105667506,For Rent,"1 Commercial Space For Rent in Jid Ali, Capita...",Discover a unique commercial space available f...,"Jid Ali, Capital Governorate",300,BHD,Yearly,Unfurnished,NaN,11 Sq. M.,Santhosh,https://images.bayut.bh/thumbnails/228310-400x...,12,Capital Governorate Other Commercial > Jid Ali...,Other Commercial
3,https://www.bayut.bh/en/property/details-10566...,Bayut - ID105667506,105667506,For Rent,"1 Commercial Space For Rent in Jid Ali, Capita...",Discover a unique commercial space available f...,"Jid Ali, Capital Governorate",300,BHD,Yearly,Unfurnished,NaN,11 Sq. M.,Santhosh,https://images.bayut.bh/thumbnails/228310-400x...,12,Capital Governorate Other Commercial > Jid Ali...,Other Commercial
4,https://www.bayut.bh/en/property/details-10569...,Bayut - ID105690991,105690991,For Rent,1 Other Commercial For Rent Manama,This commercial property located in Manama off...,"Manama, Capital Governorate",800,BHD,Yearly,Unfurnished,NaN,500 Sq. M.,Myspace Real Estate & Consulting,https://images.bayut.bh/thumbnails/226666-800x...,18,Capital Governorate Other Commercial > Manama ...,Other Commercial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1045,https://www.bayut.bh/en/property/details-10564...,Bayut - ID105648594,105648594,For Rent,"1 Bedroom Other Commercial For Rent Gufool, Ca...",Discover a unique opportunity with this commer...,"Gufool, Capital Governorate",350,BHD,Yearly,Unfurnished,NaN,110 Sq. M.,Tanmeya Investment,https://images.bayut.bh/thumbnails/213878-800x...,18,Capital Governorate Other Commercial > Gufool ...,Other Commercial
1046,https://www.bayut.bh/en/property/details-10566...,Bayut - ID105667506,105667506,For Rent,"1 Commercial Space For Rent in Jid Ali, Capita...",Discover a unique commercial space available f...,"Jid Ali, Capital Governorate",300,BHD,Yearly,Unfurnished,NaN,11 Sq. M.,Santhosh,https://images.bayut.bh/thumbnails/228310-400x...,12,Capital Governorate Other Commercial > Jid Ali...,Other Commercial
1047,https://www.bayut.bh/en/property/details-10566...,Bayut - ID105667506,105667506,For Rent,"1 Commercial Space For Rent in Jid Ali, Capita...",Discover a unique commercial space available f...,"Jid Ali, Capital Governorate",300,BHD,Yearly,Unfurnished,NaN,11 Sq. M.,Santhosh,https://images.bayut.bh/thumbnails/228310-400x...,12,Capital Governorate Other Commercial > Jid Ali...,Other Commercial
1048,https://www.bayut.bh/en/property/details-10569...,Bayut - ID105690991,105690991,For Rent,1 Other Commercial For Rent Manama,This commercial property located in Manama off...,"Manama, Capital Governorate",800,BHD,Yearly,Unfurnished,NaN,500 Sq. M.,Myspace Real Estate & Consulting,https://images.bayut.bh/thumbnails/226666-800x...,18,Capital Governorate Other Commercial > Manama ...,Other Commercial
